In [1]:
# --- BOOTSTRAP for "Save & Run All" (defines missing globals) ---
import os, glob, re, numpy as np, pandas as pd
from scipy import sparse

# 1) DATA_DIR + load data
if "DATA_DIR" not in globals():
    cand = [d for d in glob.glob("/kaggle/input/*")
            if any(k in d.lower() for k in ["polymer","neurips","open","opp"])]
    DATA_DIR = None
    for d in cand:
        if all(os.path.exists(os.path.join(d, f)) for f in ["train.csv","test.csv","sample_submission.csv"]):
            DATA_DIR = d; break
    if DATA_DIR is None:
        DATA_DIR = "/kaggle/input/neurips-open-polymer-prediction-2025"
    print("Using data dir:", DATA_DIR)

if "train" not in globals():
    train  = pd.read_csv(os.path.join(DATA_DIR,"train.csv"))
    test   = pd.read_csv(os.path.join(DATA_DIR,"test.csv"))
    sample = pd.read_csv(os.path.join(DATA_DIR,"sample_submission.csv"))

# 2) SMILES column
def _find_smiles(df):
    for c in df.columns:
        if c.lower().strip() in ("smiles","smile","smiles_str","polymer_smiles"):
            return c
    raise KeyError("SMILES column not found")
if "SMI_COL" not in globals():
    SMI_COL = _find_smiles(train)

# 3) Targets present
TARGETS = ["Tg","FFV","Tc","Density","Rg"]
def _map_targets(df):
    m = {}
    for t in TARGETS:
        hits = [c for c in df.columns if c.lower()==t.lower()]
        if hits: m[t] = hits[0]
    return m
if "tmap" not in globals():
    tmap = _map_targets(train)
if "TARGETS_CAN" not in globals():
    TARGETS_CAN = [t for t in TARGETS if t in tmap]
print("Targets present:", TARGETS_CAN)

# 4) SMILES stats (if not already computed)
ATOM_TOKENS = ["Cl","Br","Si"]
SINGLE_CHARS = list("BCNOFPSI")
AROM_CHARS   = list("cnospb")
BOND_CHARS   = list("=#")
SYMS         = list("()[]")
EXTRA        = [".","%"]
RING_DIGITS  = list("123456789")

def _smiles_stats_vector(smiles: str):
    s = smiles
    L = len(s)
    counts = {}
    for tk in ATOM_TOKENS:
        counts[tk] = len(re.findall(tk, s))
        s = s.replace(tk, " ")
    for ch in SINGLE_CHARS: counts[ch] = s.count(ch)
    for ch in AROM_CHARS:   counts[f"ar_{ch}"] = s.count(ch)
    for ch in BOND_CHARS+SYMS+EXTRA: counts[ch] = s.count(ch)
    s_orig = smiles
    for d in RING_DIGITS: counts[f"ring_{d}"] = s_orig.count(d)

    ring_total = sum(counts[f"ring_{d}"] for d in RING_DIGITS)
    aromatic_total = sum(counts[f"ar_{ch}"] for ch in AROM_CHARS)
    caps_total = sum(counts.get(ch,0) for ch in SINGLE_CHARS) + sum(counts[tk] for tk in ATOM_TOKENS)
    bonds_total = sum(counts[ch] for ch in BOND_CHARS)
    parens_total = counts["("] + counts[")"]

    feats = [
        L, ring_total, aromatic_total, caps_total, bonds_total, parens_total,
        counts["["] + counts["]"], counts["."], counts["%"],
        counts.get("Cl",0), counts.get("Br",0), counts.get("Si",0),
        counts.get("C",0), counts.get("N",0), counts.get("O",0),
        counts.get("F",0), counts.get("P",0), counts.get("S",0), counts.get("I",0),
    ]
    def ratio(x): return (x / L) if L > 0 else 0.0
    feats += [ratio(ring_total), ratio(aromatic_total), ratio(caps_total),
              ratio(bonds_total), ratio(parens_total)]
    return np.array(feats, dtype=np.float32)

STAT_NAMES = [
    "len","rings","arom","caps","bonds","parens","brackets","dots","pct",
    "Cl","Br","Si","C","N","O","F","P","S","I",
    "r_rings","r_arom","r_caps","r_bonds","r_parens"
]

def _featurize_stats(smiles_list):
    M = len(smiles_list)
    X = np.zeros((M, len(STAT_NAMES)), dtype=np.float32)
    for i, smi in enumerate(smiles_list):
        X[i,:] = _smiles_stats_vector(str(smi))
    return X

if "train_smiles" not in globals():
    train_smiles = train[SMI_COL].astype(str).values
if "test_smiles" not in globals():
    test_smiles  = test[SMI_COL].astype(str).values
if "Xstats_train" not in globals():
    Xstats_train = _featurize_stats(train_smiles)
    Xstats_test  = _featurize_stats(test_smiles)


Using data dir: /kaggle/input/neurips-open-polymer-prediction-2025
Targets present: ['Tg', 'FFV', 'Tc', 'Density', 'Rg']


In [2]:
# small hyperparam search + unlabeled SMILES augmentation, auto-select per target
import os, re, glob, gc, numpy as np, pandas as pd
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from scipy import sparse

RANDOM_STATE = 42
N_SPLITS = 5
TARGETS = ["Tg","FFV","Tc","Density","Rg"]

HAS_XGB = False
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

SUP_DIR = os.path.join(DATA_DIR, "train_supplement")
extra_smiles = []
try:
    p = os.path.join(SUP_DIR, "dataset2.csv")
    if os.path.exists(p):
        ds2 = pd.read_csv(p)
        smi_col2 = next(c for c in ds2.columns if c.lower() in ("smiles","smile","smiles_str","polymer_smiles"))
        extra_smiles = ds2[smi_col2].astype(str).tolist()
        print(f"Loaded {len(extra_smiles)} extra SMILES from dataset2.csv for TF-IDF augmentation.")
except Exception as e:
    print("dataset2.csv not used:", e)

train_smiles = train[SMI_COL].astype(str).values
test_smiles  = test[SMI_COL].astype(str).values

def compute_weights(y_df, cols):
    n = y_df[cols].notna().sum().astype(float)
    r = (y_df[cols].max() - y_df[cols].min()).astype(float).replace(0, 1.0)
    K = len(cols)
    inv_sqrt = 1/np.sqrt(n.clip(lower=1))
    norm = K * (inv_sqrt / inv_sqrt.sum())
    return (1.0/r) * norm

def approx_wmae(y_df, yhat_df, cols, weights):
    per_row = []
    for i in range(len(y_df)):
        s = 0.0
        for c in cols:
            yi = y_df.at[i,c]
            if pd.notna(yi):
                s += weights[c] * abs(yhat_df.at[i,c] - yi)
        per_row.append(s)
    denom = (y_df[cols].notna().sum(axis=1) > 0).sum()
    return float(np.sum(per_row) / max(denom,1))

y_full = train[[tmap[t] for t in TARGETS_CAN]].copy()
y_full.columns = TARGETS_CAN
weights = compute_weights(y_full, TARGETS_CAN)
print("Approx train-derived weights:", weights.to_dict())

VECT_SETTINGS = [
    dict(ngram_range=(2,5), min_df=2, max_features=200_000),  # baseline-ish
    dict(ngram_range=(2,6), min_df=1, max_features=300_000),  # longer n-grams + rarer grams
]
RIDGE_ALPHAS = [3.0, 7.0]  # small sweep
XGB_CFGS = [
    dict(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.6, reg_lambda=1.0),
    dict(n_estimators=900, learning_rate=0.03, max_depth=7, subsample=0.9, colsample_bytree=0.7, reg_lambda=1.0),
] if HAS_XGB else []

use_stats = True if 'Xstats_train' in globals() else False

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def fit_vectorizer(text_train_fold, vect_params):
    tfidf = TfidfVectorizer(analyzer="char", **vect_params)
    if len(extra_smiles):
        fit_corpus = np.concatenate([text_train_fold, np.array(extra_smiles)])
        Xtr_full = tfidf.fit_transform(fit_corpus)
        # slice back to fold size
        return tfidf, Xtr_full[:len(text_train_fold)]
    else:
        Xtr = tfidf.fit_transform(text_train_fold)
        return tfidf, Xtr

def try_config(target, vect_params, model_kind, model_params):
    mask = ~pd.isna(train[tmap[target]])
    y = train.loc[mask, tmap[target]].values
    X_text = train_smiles[mask]
    oof = np.zeros(mask.sum(), dtype=np.float32)
    pred_test = np.zeros(len(test), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_text), 1):
        X_tr_text, X_va_text = X_text[tr_idx], X_text[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        tfidf, Xtr_tfidf = fit_vectorizer(X_tr_text, vect_params)
        Xva_tfidf = tfidf.transform(X_va_text)
        Xtst_tfidf = tfidf.transform(test_smiles)

        if use_stats:
            Xtr = sparse.hstack([Xtr_tfidf, sparse.csr_matrix(Xstats_train[mask][tr_idx])]).tocsr()
            Xva = sparse.hstack([Xva_tfidf, sparse.csr_matrix(Xstats_train[mask][va_idx])]).tocsr()
            Xtst = sparse.hstack([Xtst_tfidf, sparse.csr_matrix(Xstats_test)]).tocsr()
        else:
            Xtr, Xva, Xtst = Xtr_tfidf, Xva_tfidf, Xtst_tfidf

        if model_kind == "ridge":
            model = Ridge(alpha=model_params["alpha"], solver="lsqr", random_state=RANDOM_STATE)
            model.fit(Xtr, y_tr)
            oof[va_idx] = model.predict(Xva)
            pred_test += model.predict(Xtst) / N_SPLITS

        elif model_kind == "xgb":
            model = xgb.XGBRegressor(
                objective="reg:absoluteerror",
                eval_metric="mae",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                early_stopping_rounds=50,
                **model_params
            )
            model.fit(Xtr, y_tr, eval_set=[(Xva, y_va)], verbose=False)
            oof[va_idx] = model.predict(Xva)
            pred_test += model.predict(Xtst) / N_SPLITS

        del tfidf, Xtr_tfidf, Xva_tfidf, Xtst_tfidf
        gc.collect()

    mae = mean_absolute_error(y, oof)
    w_contrib = float(weights[target] * mae)
    return oof, pred_test, w_contrib, mae

OOF_sel = pd.DataFrame(index=train.index, columns=TARGETS_CAN, dtype=float)
PRED_sel = pd.DataFrame({"id": test["id"].values})
for t in TARGETS_CAN: PRED_sel[t] = 0.0
summary = []

for target in TARGETS_CAN:
    print(f"\n=== {target}: searching small grid ===")
    best = None
    store = {}
    for vp in VECT_SETTINGS:
        for a in RIDGE_ALPHAS:
            cfg_name = f"ridge|ngr{vp['ngram_range']}_min{vp['min_df']}_a{a}"
            oof, pred, w, mae = try_config(target, vp, "ridge", {"alpha": a})
            store[cfg_name] = (oof, pred, w, mae)
            print(f"  {cfg_name:<40}  w*MAE={w:.6f}  MAE={mae:.6f}")
    for vp in VECT_SETTINGS:
        for j, xcfg in enumerate(XGB_CFGS):
            cfg_name = f"xgb|ngr{vp['ngram_range']}_min{vp['min_df']}_set{j+1}"
            oof, pred, w, mae = try_config(target, vp, "xgb", xcfg)
            store[cfg_name] = (oof, pred, w, mae)
            print(f"  {cfg_name:<40}  w*MAE={w:.6f}  MAE={mae:.6f}")

    best_cfg = min(store.items(), key=lambda kv: kv[1][2])
    name, (oof, pred, w, mae) = best_cfg
    print(f"  -> selected: {name}  (w*MAE={w:.6f}, MAE={mae:.6f})")
    mask = ~pd.isna(train[tmap[target]])
    OOF_sel.loc[mask, target] = oof
    PRED_sel[target] = pred

    summary.append({"target": target, "selected": name, "selected_w*MAE": w, "selected_MAE": mae})

summary_df = pd.DataFrame(summary)
print("\nSelection summary:\n", summary_df)

overall = approx_wmae(y_full, OOF_sel[TARGETS_CAN], TARGETS_CAN, weights)
print("\nApprox overall wMAE (selected):", round(overall, 6))

sub = pd.DataFrame({
    "id": test["id"].values,
    "Tg": PRED_sel["Tg"] if "Tg" in PRED_sel else 0.0,
    "FFV": PRED_sel["FFV"] if "FFV" in PRED_sel else 0.0,
    "Tc": PRED_sel["Tc"] if "Tc" in PRED_sel else 0.0,
    "Density": PRED_sel["Density"] if "Density" in PRED_sel else 0.0,
    "Rg": PRED_sel["Rg"] if "Rg" in PRED_sel else 0.0,
})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved /kaggle/working/submission.csv")


Loaded 7208 extra SMILES from dataset2.csv for TF-IDF augmentation.
Approx train-derived weights: {'Tg': 0.00205237749659811, 'FFV': 0.6239248659724905, 'Tc': 2.2199754352943795, 'Density': 1.0640941761320573, 'Rg': 0.046558118429742154}

=== Tg: searching small grid ===
  ridge|ngr(2, 5)_min2_a3.0                 w*MAE=0.114291  MAE=55.687334
  ridge|ngr(2, 5)_min2_a7.0                 w*MAE=0.117981  MAE=57.484841
  ridge|ngr(2, 6)_min1_a3.0                 w*MAE=0.114512  MAE=55.794714
  ridge|ngr(2, 6)_min1_a7.0                 w*MAE=0.118353  MAE=57.666291
  xgb|ngr(2, 5)_min2_set1                   w*MAE=0.109611  MAE=53.406902
  xgb|ngr(2, 5)_min2_set2                   w*MAE=0.111279  MAE=54.219433
  xgb|ngr(2, 6)_min1_set1                   w*MAE=0.110755  MAE=53.964351
  xgb|ngr(2, 6)_min1_set2                   w*MAE=0.110995  MAE=54.081151
  -> selected: xgb|ngr(2, 5)_min2_set1  (w*MAE=0.109611, MAE=53.406902)

=== FFV: searching small grid ===
  ridge|ngr(2, 5)_min2_a3.0  